In [12]:
# CELL 1 - Define input and output paths
# Reads cleaned data and lookup CSV, then writes final curated output.

storage_account = "straccajs"
processed_clean_path = f"abfss://processed@{storage_account}.dfs.core.windows.net/yellow_clean/"
zone_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/lookup/taxi_zone_lookup.csv"
curated_path = f"abfss://curated@{storage_account}.dfs.core.windows.net/yellow_trips/"

StatementMeta(sparkpool1, 1, 8, Finished, Available, Finished, False)

In [14]:
# CELL 2 - Read cleaned trips and zone lookup file
# Loads the cleaned January dataset and the taxi zone reference table.

from pyspark.sql import functions as F

df = spark.read.parquet(processed_clean_path)
zones = spark.read.option("header", True).csv(zone_path)

zones.printSchema()
zones.show(5)

StatementMeta(sparkpool1, 1, 10, Finished, Available, Finished, False)

root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [15]:
# CELL 3 - Join pickup and dropoff zone information
# Enriches the cleaned trip data by replacing location IDs with readable
# borough and zone names for both pickup and dropoff.

df_enriched = (df
    .join(zones.alias("pu"), df.PULocationID == F.col("pu.LocationID"), "left")
    .withColumnRenamed("Borough", "pickup_borough")
    .withColumnRenamed("Zone", "pickup_zone")
    .drop("LocationID", "service_zone")
    .join(zones.alias("do"), df.DOLocationID == F.col("do.LocationID"), "left")
    .withColumnRenamed("Borough", "dropoff_borough")
    .withColumnRenamed("Zone", "dropoff_zone")
    .drop("LocationID", "service_zone")
)

df_enriched.select("pickup_borough", "pickup_zone", "dropoff_borough", "dropoff_zone").show(5)


StatementMeta(sparkpool1, 1, 11, Finished, Available, Finished, False)

+--------------+--------------------+---------------+--------------+
|pickup_borough|         pickup_zone|dropoff_borough|  dropoff_zone|
+--------------+--------------------+---------------+--------------+
|     Manhattan| UN/Turtle Bay South|            EWR|Newark Airport|
|     Manhattan|       Midtown North|            EWR|Newark Airport|
|     Manhattan|    Garment District|            EWR|Newark Airport|
|     Manhattan| UN/Turtle Bay South|            EWR|Newark Airport|
|     Manhattan|Financial Distric...|            EWR|Newark Airport|
+--------------+--------------------+---------------+--------------+
only showing top 5 rows



In [16]:
# CELL 4 - Apply anonymisation steps
# Reduces timestamp precision and hashes VendorID to lower re-identification risk.
# These are defensive privacy measures because the dataset has no direct PII.

df_anon = (df_enriched
    .withColumn("pickup_hour_bucket", F.date_trunc("hour", "tpep_pickup_datetime"))
    .withColumn("dropoff_hour_bucket", F.date_trunc("hour", "tpep_dropoff_datetime"))
    .withColumn("vendor_hash", F.sha2(F.col("VendorID").cast("string"), 256))
    .drop("VendorID")
)

StatementMeta(sparkpool1, 1, 12, Finished, Available, Finished, False)

In [17]:
# CELL 5 - Select curated columns and write final output
# Builds the final analytics-ready dataset and writes it to /curated/yellow_trips/
# partitioned by pickup_month.

df_curated = df_anon.select(
    "vendor_hash", "pickup_hour_bucket", "dropoff_hour_bucket",
    "passenger_count", "trip_distance", "trip_duration_min",
    "pickup_borough", "pickup_zone", "dropoff_borough", "dropoff_zone",
    "fare_amount", "tip_amount", "total_amount", "payment_type",
    "pickup_hour", "pickup_day", "is_weekend", "pickup_month"
)

(df_curated.write
    .mode("overwrite")
    .partitionBy("pickup_month")
    .parquet(curated_path))

print(f"Curated rows: {df_curated.count():,}")
df_curated.show(5)

StatementMeta(sparkpool1, 1, 13, Finished, Available, Finished, False)

Curated rows: 2,724,063
+--------------------+-------------------+-------------------+---------------+-------------+------------------+--------------+--------------------+---------------+--------------+-----------+----------+------------+------------+-----------+----------+----------+------------+
|         vendor_hash| pickup_hour_bucket|dropoff_hour_bucket|passenger_count|trip_distance| trip_duration_min|pickup_borough|         pickup_zone|dropoff_borough|  dropoff_zone|fare_amount|tip_amount|total_amount|payment_type|pickup_hour|pickup_day|is_weekend|pickup_month|
+--------------------+-------------------+-------------------+---------------+-------------+------------------+--------------+--------------------+---------------+--------------+-----------+----------+------------+------------+-----------+----------+----------+------------+
|d4735e3a265e16eee...|2024-01-01 02:00:00|2024-01-01 03:00:00|              1|        19.34|             45.85|     Manhattan| UN/Turtle Bay South|    